# Aula 13: Métodos Iterativos

## Gauss-Jacobi e Gauss-Seidel

## Ideia

Para resolver $Ax=b$, partimos de uma aproximação inicial $x^{(0)}$ e construímos novas aproximações até satisfazer um critério de parada.

$$d^{(k)}=\frac{\max_i|x_i^{(k)}-x_i^{(k-1)}|}{\max_i|x_i^{(k)}|}<\varepsilon$$

No **Gauss-Jacobi**, todas as incógnitas usam os valores da iteração anterior. No **Gauss-Seidel**, usamos imediatamente os valores recém-calculados.

## Algoritmo: Gauss-Jacobi

```text
Entrada: A (n × n), b (n), x inicial (n), ε, max_iter

Para k de 1 até max_iter faça
    anterior = cópia de x
    Para i de 1 até n faça
        s = 0
        Para j de 1 até n faça
            Se j ≠ i então
                s = s + A[i,j] * anterior[j]
            Fim-Se
        Fim-Para
        x[i] = (b[i] - s) / A[i,i]
    Fim-Para
    Se erro_relativo(x, anterior) < ε então
        Retorne x
    Fim-Se
Fim-Para
```

In [ ]:
def erro_relativo(x, anterior):
    numerador = max(abs(x[i] - anterior[i]) for i in range(len(x)))
    denominador = max(abs(v) for v in x)
    return numerador / denominador if denominador != 0 else numerador


def gauss_jacobi(A, b, x0, eps=0.05, max_iter=100):
    n = len(b)
    x = x0[:]
    historico = [(0, x[:], None)]
    for k in range(1, max_iter + 1):
        anterior = x[:]
        for i in range(n):
            s = sum(A[i][j] * anterior[j] for j in range(n) if j != i)
            x[i] = (b[i] - s) / A[i][i]
        d = erro_relativo(x, anterior)
        historico.append((k, x[:], d))
        if d < eps:
            break
    return x, historico

In [ ]:
def mostrar_iteracoes(historico):
    for k, x, d in historico:
        valores = ', '.join(f'{v:.4f}' for v in x)
        erro = '—' if d is None else f'{d:.4f}'
        print(f'k={k}: x=({valores}) | d={erro}')


def criterio_linhas(A):
    return [sum(abs(A[i][j]) for j in range(len(A)) if j != i) / abs(A[i][i])
            for i in range(len(A))]


def sassenfeld(A):
    n = len(A)
    beta = [0.0] * n
    for i in range(n):
        soma = sum(abs(A[i][j]) * beta[j] for j in range(i))
        soma += sum(abs(A[i][j]) for j in range(i + 1, n))
        beta[i] = soma / abs(A[i][i])
    return beta

## Exemplo 1: Gauss-Jacobi

$$\begin{cases}10x_1+2x_2+x_3=7\\x_1+5x_2+x_3=-8\\2x_1+3x_2+10x_3=6\end{cases}$$

$$x^{(0)}=\begin{bmatrix}0.7\\-1.6\\0.6\end{bmatrix},\qquad \varepsilon=0.05$$

$$\begin{cases}
x_1^{(k+1)}=(7-2x_2^{(k)}-x_3^{(k)})/10\\
x_2^{(k+1)}=(-8-x_1^{(k)}-x_3^{(k)})/5\\
x_3^{(k+1)}=(6-2x_1^{(k)}-3x_2^{(k)})/10
\end{cases}$$

In [2]:
A_j = [[10, 2, 1], [1, 5, 1], [2, 3, 10]]
b_j = [7, -8, 6]
x_j, hist_j = gauss_jacobi(A_j, b_j, [0.7, -1.6, 0.6], eps=0.05)
mostrar_iteracoes(hist_j)
print('Critério das linhas:', criterio_linhas(A_j))

k=0: x=(0.7000, -1.6000, 0.6000) | d=—
k=1: x=(0.9600, -1.8600, 0.9400) | d=0.1828
k=2: x=(0.9780, -1.9800, 0.9660) | d=0.0606
k=3: x=(0.9994, -1.9888, 0.9984) | d=0.0163
Critério das linhas: [0.3, 0.4, 0.5]


## Exercício: Gauss-Jacobi

$$\begin{cases}4x_1+x_2+x_3=6\\x_1+5x_2+x_3=7\\x_1+x_2+6x_3=8\end{cases}$$

$$x^{(0)}=\begin{bmatrix}0\\0\\0\end{bmatrix},\qquad\varepsilon=0.05$$

Verifique o critério das linhas, determine as fórmulas iterativas e calcule as aproximações até satisfazer o critério de parada relativo.

In [3]:
A_ex = [[4, 1, 1], [1, 5, 1], [1, 1, 6]]
b_ex = [6, 7, 8]
x, historico = gauss_jacobi(A_ex, b_ex, [0, 0, 0], eps=0.05)
mostrar_iteracoes(historico)
print('Critério das linhas:', criterio_linhas(A_ex))
print('Solução aproximada:', x)

k=0: x=(0.0000, 0.0000, 0.0000) | d=—
k=1: x=(1.5000, 1.4000, 1.3333) | d=1.0000
k=2: x=(0.8167, 0.8333, 0.8500) | d=0.8039
k=3: x=(1.0792, 1.0667, 1.0583) | d=0.2432
k=4: x=(0.9688, 0.9725, 0.9757) | d=0.1132
k=5: x=(1.0130, 1.0111, 1.0098) | d=0.0436
Critério das linhas: [0.5, 0.4, 0.3333333333333333]
Solução aproximada: [1.0129513888888888, 1.011111111111111, 1.0097916666666666]


## Algoritmo: Gauss-Seidel

```text
Entrada: A (n × n), b (n), x inicial (n), ε, max_iter

Para k de 1 até max_iter faça
    anterior = cópia de x
    Para i de 1 até n faça
        s = 0
        Para j de 1 até n faça
            Se j ≠ i então
                s = s + A[i,j] * x[j]
            Fim-Se
        Fim-Para
        x[i] = (b[i] - s) / A[i,i]
    Fim-Para
    Se erro_relativo(x, anterior) < ε então
        Retorne x
    Fim-Se
Fim-Para
```

**Diferença:** `x[j]` é atualizado durante a própria iteração.

In [4]:
def gauss_seidel(A, b, x0, eps=0.05, max_iter=100):
    n = len(b)
    x = x0[:]
    historico = [(0, x[:], None)]
    for k in range(1, max_iter + 1):
        anterior = x[:]
        for i in range(n):
            s = sum(A[i][j] * x[j] for j in range(n) if j != i)
            x[i] = (b[i] - s) / A[i][i]
        d = erro_relativo(x, anterior)
        historico.append((k, x[:], d))
        if d < eps:
            break
    return x, historico

## Exemplo 2: Gauss-Seidel

$$\begin{cases}5x_1+x_2+x_3=5\\3x_1+4x_2+x_3=6\\3x_1+3x_2+6x_3=0\end{cases}$$

$$x^{(0)}=\begin{bmatrix}0\\0\\0\end{bmatrix},\qquad\varepsilon=0.05$$

$$\begin{cases}
x_1^{(k+1)}=1-0.2x_2^{(k)}-0.2x_3^{(k)}\\
x_2^{(k+1)}=1.5-0.75x_1^{(k+1)}-0.25x_3^{(k)}\\
x_3^{(k+1)}=-0.5x_1^{(k+1)}-0.5x_2^{(k+1)}
\end{cases}$$

In [5]:
A_s = [[5, 1, 1], [3, 4, 1], [3, 3, 6]]
b_s = [5, 6, 0]
x_s, hist_s = gauss_seidel(A_s, b_s, [0, 0, 0], eps=0.05)
mostrar_iteracoes(hist_s)
print('Sassenfeld:', sassenfeld(A_s))

k=0: x=(0.0000, 0.0000, 0.0000) | d=—
k=1: x=(1.0000, 0.7500, -0.8750) | d=1.0000
k=2: x=(1.0250, 0.9500, -0.9875) | d=0.1951
k=3: x=(1.0075, 0.9913, -0.9994) | d=0.0409
Sassenfeld: [0.4, 0.55, 0.4750000000000001]


## Exemplo 3: Critério de Sassenfeld

$$A=\begin{bmatrix}
1&0.5&-0.1&0.1\\
0.2&1&-0.2&-0.1\\
-0.1&-0.2&1&0.2\\
0.1&0.3&0.2&1
\end{bmatrix}$$

Calculamos $\beta_1,\ldots,\beta_4$. Se $\max_i\beta_i<1$, a convergência está garantida.

In [6]:
A_sass = [[1, 0.5, -0.1, 0.1], [0.2, 1, -0.2, -0.1],
          [-0.1, -0.2, 1, 0.2], [0.1, 0.3, 0.2, 1]]
beta = sassenfeld(A_sass)
print('Betas:', [round(v, 4) for v in beta])
print('Convergência garantida:', max(beta) < 1)

Betas: [0.7, 0.44, 0.358, 0.2736]
Convergência garantida: True


## Exemplo 4: Critérios das linhas e de Sassenfeld

$$\begin{cases}3x_1+x_3=3\\x_1-x_2=1\\3x_1+x_2+2x_3=9\end{cases}$$

O critério das linhas não é satisfeito, mas o critério de Sassenfeld garante a convergência.

In [7]:
A_comp = [[3, 0, 1], [1, -1, 0], [3, 1, 2]]
print('Linhas:', criterio_linhas(A_comp))
print('Sassenfeld:', sassenfeld(A_comp))

Linhas: [0.3333333333333333, 1.0, 2.0]
Sassenfeld: [0.3333333333333333, 0.3333333333333333, 0.6666666666666666]


## Exercício: Gauss-Seidel

$$\begin{cases}4x_1+x_2+x_3=6\\x_1+5x_2+x_3=7\\x_1+x_2+6x_3=8\end{cases}$$

$$x^{(0)}=\begin{bmatrix}0\\0\\0\end{bmatrix},\qquad\varepsilon=0.05$$

Verifique Sassenfeld, determine as fórmulas iterativas e calcule as aproximações até satisfazer o critério de parada relativo.

In [8]:
x, historico = gauss_seidel(A_ex, b_ex, [0, 0, 0], eps=0.05)
mostrar_iteracoes(historico)
print('Sassenfeld:', sassenfeld(A_ex))
print('Solução aproximada:', x)

k=0: x=(0.0000, 0.0000, 0.0000) | d=—
k=1: x=(1.5000, 1.1000, 0.9000) | d=1.0000
k=2: x=(1.0000, 1.0200, 0.9967) | d=0.4902
k=3: x=(0.9958, 1.0015, 1.0004) | d=0.0185
Sassenfeld: [0.5, 0.3, 0.13333333333333333]
Solução aproximada: [0.9958333333333333, 1.0015, 1.0004444444444445]
